# 05 - Gold Layer - Supplier Dimension

Create a business-ready Supplier Dimension from the validated Silver supplier data.

**Source:** `end-to-end_pipeline.silver.suppliers`
**Target:** `end-to-end_pipeline.gold.dim_supplier`

**Model Role:** Dimension Table
**Business Key:** `supplier_id`
**Approach:** Profile → Inspect → Transform → Validate

**Purpose:**
Provide supplier attributes for analyzing product and sales performance by supplier, supplier category, country, lead time, rating, contract period, and active status.

## Cell 1 - Profile Silver Supplier Data

**Description:**
Confirm that the Silver Suppliers table is ready to become a Gold dimension by checking key uniqueness, row count, and availability of the business attributes required for analysis.


In [0]:
%sql

-- ============================================================
-- CELL 1: PROFILE SILVER SUPPLIERS FOR GOLD MODELING
-- Purpose: Confirm dimension grain, key uniqueness,
--          and availability of business attributes
-- ============================================================

SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT supplier_id) AS distinct_supplier_ids,
    COUNT(*) - COUNT(DISTINCT supplier_id) AS duplicate_supplier_ids,

    SUM(CASE WHEN supplier_id IS NULL THEN 1 ELSE 0 END)
        AS null_supplier_ids,

    COUNT(DISTINCT supplier_category)
        AS supplier_categories,

    COUNT(DISTINCT country)
        AS countries,

    COUNT(DISTINCT active_status)
        AS active_statuses,

    MIN(lead_time_days)
        AS minimum_lead_time_days,

    MAX(lead_time_days)
        AS maximum_lead_time_days

FROM `end-to-end_pipeline`.silver.suppliers;

total_rows,distinct_supplier_ids,duplicate_supplier_ids,null_supplier_ids,supplier_categories,countries,active_statuses,minimum_lead_time_days,maximum_lead_time_days
40,40,0,0,4,8,2,3,30


## Cell 2 - Inspect Supplier Business Attributes

**Description:**
Review the main Supplier attributes that will be useful for business analysis, especially supplier category, country, active status, and rating distribution.

In [0]:
%sql

-- ============================================================
-- CELL 2: INSPECT SUPPLIER BUSINESS ATTRIBUTES
-- Purpose: Review supplier distribution across business groups
-- ============================================================

SELECT
    supplier_category,
    country,
    active_status,
    COUNT(*) AS supplier_count,
    ROUND(AVG(supplier_rating), 2) AS average_supplier_rating

FROM `end-to-end_pipeline`.silver.suppliers

GROUP BY
    supplier_category,
    country,
    active_status

ORDER BY
    supplier_category,
    country,
    active_status;

supplier_category,country,active_status,supplier_count,average_supplier_rating
Accessories,Germany,Active,2,3.30
Accessories,Italy,Active,1,2.80
Accessories,Italy,Inactive,1,4.80
Accessories,Netherlands,Active,2,4.00
Accessories,Netherlands,Inactive,1,3.80
Accessories,Poland,Active,2,4.00
Accessories,Spain,Active,1,4.00
Accessories,Sweden,Active,2,4.05
Electronics,France,Active,2,3.95
Electronics,Netherlands,Active,1,3.80


## Cell 3 - Transform Silver → Gold Supplier Dimension

**Description:**
Create the Gold Supplier Dimension at **one row per supplier**. The Silver layer has already handled cleaning and validation, so this step only exposes the descriptive supplier attributes needed for business analytics.

In [0]:
%sql

-- ============================================================
-- CELL 3: CREATE GOLD SUPPLIER DIMENSION
-- Grain: One row per supplier
-- Business Key: supplier_id
-- ============================================================

CREATE OR REPLACE TABLE `end-to-end_pipeline`.gold.dim_supplier AS

SELECT
    supplier_id,
    supplier_name,
    supplier_category,
    country,
    lead_time_days,
    supplier_rating,
    contract_start_date,
    YEAR(contract_start_date) AS contract_year,
    active_status

FROM `end-to-end_pipeline`.silver.suppliers;

num_affected_rows,num_inserted_rows


## Cell 3a - Add Business Metadata

**Description:**
Add table and column comments to improve discoverability in Genie and Databricks dashboards. These descriptions help users understand the business meaning of each attribute.

In [0]:
%sql

-- ============================================================
-- CELL 3a: ADD BUSINESS METADATA TO SUPPLIER DIMENSION
-- Purpose: Improve discoverability for Genie and dashboards
-- ============================================================

COMMENT ON TABLE `end-to-end_pipeline`.gold.dim_supplier IS 
'Supplier dimension providing vendor attributes for product sourcing and supply chain analysis. One row per supplier.';

ALTER TABLE `end-to-end_pipeline`.gold.dim_supplier 
  ALTER COLUMN supplier_id COMMENT 'Unique supplier identifier (business key)';

ALTER TABLE `end-to-end_pipeline`.gold.dim_supplier 
  ALTER COLUMN supplier_name COMMENT 'Display name of the supplier';

ALTER TABLE `end-to-end_pipeline`.gold.dim_supplier 
  ALTER COLUMN supplier_category COMMENT 'Supplier classification category (e.g., Manufacturer, Distributor)';

ALTER TABLE `end-to-end_pipeline`.gold.dim_supplier 
  ALTER COLUMN country COMMENT 'Country where the supplier is located';

ALTER TABLE `end-to-end_pipeline`.gold.dim_supplier 
  ALTER COLUMN lead_time_days COMMENT 'Average lead time in days for supplier deliveries';

ALTER TABLE `end-to-end_pipeline`.gold.dim_supplier 
  ALTER COLUMN supplier_rating COMMENT 'Supplier performance rating';

ALTER TABLE `end-to-end_pipeline`.gold.dim_supplier 
  ALTER COLUMN contract_start_date COMMENT 'Date when the supplier contract became active';

ALTER TABLE `end-to-end_pipeline`.gold.dim_supplier 
  ALTER COLUMN contract_year COMMENT 'Year the supplier contract started (derived from contract_start_date)';

ALTER TABLE `end-to-end_pipeline`.gold.dim_supplier 
  ALTER COLUMN active_status COMMENT 'Supplier active status indicator';

## Cell 4 - Validate Gold Supplier Dimension

**Description:**
Validate that the Supplier Dimension contains one row per supplier, preserves the Silver source coverage, and maintains the required business attributes.

In [0]:
%sql

-- ============================================================
-- CELL 4: VALIDATE GOLD SUPPLIER DIMENSION
-- Purpose: Confirm dimension grain, key integrity,
--          and Silver → Gold completeness
-- ============================================================

WITH validation AS (

    SELECT
        COUNT(*) AS total_rows,

        COUNT(DISTINCT supplier_id)
            AS distinct_supplier_ids,

        COUNT(*) - COUNT(DISTINCT supplier_id)
            AS duplicate_supplier_ids,

        SUM(
            CASE
                WHEN supplier_id IS NULL THEN 1
                ELSE 0
            END
        ) AS null_supplier_ids,

        SUM(
            CASE
                WHEN supplier_name IS NULL THEN 1
                ELSE 0
            END
        ) AS null_supplier_names,

        SUM(
            CASE
                WHEN supplier_category IS NULL THEN 1
                ELSE 0
            END
        ) AS null_supplier_categories,

        SUM(
            CASE
                WHEN country IS NULL THEN 1
                ELSE 0
            END
        ) AS null_countries,

        SUM(
            CASE
                WHEN active_status IS NULL THEN 1
                ELSE 0
            END
        ) AS null_active_status,

        SUM(
            CASE
                WHEN contract_year IS NULL THEN 1
                ELSE 0
            END
        ) AS null_contract_years

    FROM `end-to-end_pipeline`.gold.dim_supplier
),

source_check AS (

    SELECT
        COUNT(*) AS silver_rows

    FROM `end-to-end_pipeline`.silver.suppliers
)

SELECT
    v.*,
    s.silver_rows,

    CASE
        WHEN v.total_rows = s.silver_rows
            AND v.distinct_supplier_ids = v.total_rows
            AND v.duplicate_supplier_ids = 0
            AND v.null_supplier_ids = 0
            AND v.null_supplier_names = 0
            AND v.null_supplier_categories = 0
            AND v.null_countries = 0
            AND v.null_active_status = 0
            AND v.null_contract_years = 0
        THEN 'PASS'
        ELSE 'FAIL'
    END AS validation_status

FROM validation v
CROSS JOIN source_check s;

total_rows,distinct_supplier_ids,duplicate_supplier_ids,null_supplier_ids,null_supplier_names,null_supplier_categories,null_countries,null_active_status,null_contract_years,silver_rows,validation_status
40,40,0,0,0,0,0,0,0,40,PASS
